# Marketplace Funnel & Activation Analysis

This notebook contains the reproducible analytical trail behind the portfolio summary. It separates observed patterns from hypotheses and uses user-level denominators unless stated otherwise.

In [1]:
from pathlib import Path
import json, sys
import duckdb
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import statsmodels.formula.api as smf
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.metrics import user_funnel, funnel_summary
RAW = ROOT / 'data/raw'
users = pd.read_csv(RAW/'users.csv', parse_dates=['signup_date'])
events = pd.read_csv(RAW/'events.csv', parse_dates=['event_timestamp'])
orders = pd.read_csv(RAW/'orders.csv', parse_dates=['order_timestamp'])
len(users), len(events), len(orders)

(40000, 468253, 25472)

## 1. Metric contract

- **User purchase rate:** unique purchasers / acquired users. It measures acquisition quality without rewarding repeat orders.
- **Exploration rate:** users with a listing or offer view / acquired users.
- **Checkout completion:** purchasing users / users who started checkout. This isolates the post-checkout step.
- **60-day repeat rate:** users with a second purchase within 60 days / first-time buyers with a full 60-day observation window.

The funnel is non-strict at user level: a user is counted if they reached a stage at least once. Session-level checkout analysis is used separately when diagnosing friction.

In [2]:
flags = user_funnel(events)
funnel = funnel_summary(flags)
funnel.style.format({'share_of_acquired_users':'{:.1%}', 'step_conversion':'{:.1%}'})

,stage,users,share_of_acquired_users,step_conversion
0,session_start,39885,100.0%,100.0%
1,listing_view,37686,94.5%,94.5%
2,offer_view,29148,73.1%,77.3%
3,checkout_start,21024,52.7%,72.1%
4,purchase,16257,40.8%,77.3%


## 2. Acquisition quality

Traffic volume and downstream value are evaluated separately. A growing channel can increase the numerator of visits while reducing the portfolio's average conversion rate through mix shift.

In [3]:
profile = users.merge(flags, on='user_id', how='left')
profile[['session_start','listing_view','offer_view','checkout_start','purchase']] = profile[['session_start','listing_view','offer_view','checkout_start','purchase']].fillna(0).astype(int)
value = orders.groupby('user_id').agg(orders=('order_id','nunique'), revenue=('order_value','sum')).reset_index()
profile = profile.merge(value, on='user_id', how='left').fillna({'orders':0,'revenue':0})
channel = profile.groupby('acquisition_channel').agg(acquired_users=('user_id','size'), purchasers=('purchase','sum'), revenue=('revenue','sum')).assign(purchase_rate=lambda x:x.purchasers/x.acquired_users, revenue_per_user=lambda x:x.revenue/x.acquired_users).sort_values('purchase_rate', ascending=False)
channel.style.format({'purchase_rate':'{:.1%}', 'revenue_per_user':'£{:.2f}', 'revenue':'£{:,.0f}'})

,acquired_users,purchasers,revenue,purchase_rate,revenue_per_user
acquisition_channel,,,,,
direct,8432,4349,"£658,038",51.6%,£78.04
organic_search,9627,4460,"£628,193",46.3%,£65.25
referral,4852,2111,"£281,593",43.5%,£58.04
paid_search,7486,3161,"£428,352",42.2%,£57.22
paid_social,9603,2176,"£220,568",22.7%,£22.97


In [4]:
profile['signup_month'] = profile.signup_date.dt.to_period('M').astype(str)
cohorts = profile.groupby(['signup_month','acquisition_channel']).agg(users=('user_id','size'), purchase_rate=('purchase','mean')).reset_index()
cohorts.pivot(index='signup_month', columns='acquisition_channel', values='users')

acquisition_channel,direct,organic_search,paid_search,paid_social,referral
signup_month,,,,,
2024-01,920,1075,796,390,598
2024-02,1308,1465,1177,728,758
2024-03,1613,1619,1337,1291,843
2024-04,1551,1810,1346,1768,898
2024-05,1565,1819,1468,2353,939
2024-06,1475,1839,1362,3073,816


**Interpretation.** The aggregate cohort decline should not be read as every channel deteriorating. The channel mix changes. This is a composition effect and motivates channel-standardized monitoring.

## 3. Device friction after checkout

Comparing purchase rates by device mixes acquisition intent with interface effects. Checkout completion provides a closer behavioral comparison, although it remains observational.

In [5]:
sessions = events.assign(checkout=events.event_name.eq('checkout_start'), purchase_event=events.event_name.eq('purchase')).groupby(['session_id','device_type']).agg(checkout=('checkout','max'), purchase=('purchase_event','max')).reset_index()
checkout = sessions[sessions.checkout].groupby('device_type').agg(checkouts=('checkout','sum'), purchases=('purchase','sum')).assign(completion=lambda x:x.purchases/x.checkouts)
checkout.style.format({'completion':'{:.1%}'})

,checkouts,purchases,completion
device_type,,,
desktop,13170,11122,84.4%
mobile,21665,14350,66.2%


**What the data shows:** mobile checkout starters finish less often. **Hypotheses:** form friction, payment-method coverage, slower performance, or a different context of use. Device assignment is not randomized, so the gap supports prioritizing instrumentation and an experiment—not a causal claim.

## 4. Early behavioral signal: offer-depth sensitivity

First-session offer depth is grouped as 0, 1, 2, and 3+ views rather than assuming a threshold. The outcome is a purchase **after the first session ends and within 14 days of its start**. Users whose first session begins fewer than 14 days before data collection ends are excluded, so incomplete follow-up is not treated as failure.

In [6]:
bounds = events.groupby(['user_id','session_id']).event_timestamp.agg(first_session_start='min', first_session_end='max').reset_index()
first_sessions = bounds.sort_values('first_session_start').drop_duplicates('user_id')
first_events = events.merge(first_sessions[['user_id','session_id']], on=['user_id','session_id'])
early = first_events.groupby('user_id').event_name.apply(lambda x:(x=='offer_view').sum()).rename('offer_views_first_session')
candidate = profile.merge(first_sessions[['user_id','first_session_start','first_session_end']], on='user_id').merge(early, on='user_id')
candidate = candidate[candidate.first_session_start <= events.event_timestamp.max()-pd.Timedelta(days=14)].copy()
candidate['offer_depth_group'] = pd.cut(candidate.offer_views_first_session, [-1,0,1,2,np.inf], labels=['0','1','2','3+'])
window = orders.merge(candidate[['user_id','first_session_start','first_session_end']], on='user_id')
window = window[(window.order_timestamp > window.first_session_end) & (window.order_timestamp <= window.first_session_start+pd.Timedelta(days=14))]
future = window.groupby('user_id').order_id.nunique().rename('future_orders_14d')
candidate = candidate.merge(future, on='user_id', how='left').fillna({'future_orders_14d':0})
candidate['subsequent_purchase_14d'] = candidate.future_orders_14d.gt(0).astype(int)
depth = candidate.groupby('offer_depth_group', observed=True).agg(users=('user_id','size'), subsequent_purchase_rate_14d=('subsequent_purchase_14d','mean'))
depth.assign(user_share=depth.users/depth.users.sum()).style.format({'subsequent_purchase_rate_14d':'{:.1%}','user_share':'{:.1%}'})

,users,subsequent_purchase_rate_14d,user_share
offer_depth_group,,,
0,16657,6.1%,46.4%
1,12548,6.8%,34.9%
2,4968,7.9%,13.8%
3+,1738,8.2%,4.8%


In [7]:
model = smf.logit("subsequent_purchase_14d ~ C(offer_depth_group, Treatment(reference='0')) + C(acquisition_channel) + C(device_type) + C(country) + C(signup_month)", data=candidate).fit(disp=False)
terms = model.params.index[model.params.index.str.contains('offer_depth_group')]
pd.DataFrame({'odds_ratio':np.exp(model.params[terms]), 'ci_low':np.exp(model.conf_int().loc[terms,0]), 'ci_high':np.exp(model.conf_int().loc[terms,1]), 'p_value':model.pvalues[terms]})

,odds_ratio,ci_low,ci_high,p_value
"C(offer_depth_group, Treatment(reference='0'))[T.1]",1.102805,1.003356,1.212111,0.042412
"C(offer_depth_group, Treatment(reference='0'))[T.2]",1.260478,1.115812,1.423901,0.000198
"C(offer_depth_group, Treatment(reference='0'))[T.3+]",1.296504,1.078254,1.558928,0.005761


**Interpretation.** Subsequent purchase rises gradually from 6.1% at zero views to 6.8%, 7.9%, and 8.2%. Adjustment for channel, device, country, and cohort reduces the contrast but preserves the ordering (odds ratios versus zero views: 1.10, 1.26, and 1.30). There is no clear cliff at two views: 2 and 3+ are similar, so `2+` remains a convenient candidate segmentation rather than an empirically unique activation threshold.

**Synthetic construction check.** The generator uses offer depth in same-session checkout probability and uses a persistent latent-intent variable for both exploration and later sessions. Same-session purchases are excluded here, removing the direct mechanical path, but the shared planted intent still induces association. This demonstrates an evaluation workflow; it does not validate a real-world mechanism.

## 5. Category and retention checks

In [8]:
category_sessions = events.groupby(['session_id','category']).event_name.agg(list).reset_index()
category_sessions['explored'] = category_sessions.event_name.apply(lambda x: 'listing_view' in x or 'offer_view' in x)
category_sessions['purchased'] = category_sessions.event_name.apply(lambda x: 'purchase' in x)
category_sessions.groupby('category').agg(exploring_sessions=('explored','sum'), purchases=('purchased','sum')).assign(purchase_per_exploration=lambda x:x.purchases/x.exploring_sessions).sort_values('purchase_per_exploration')

,exploring_sessions,purchases,purchase_per_exploration
category,,,
fashion,19137,4442,0.232116
collectibles,16266,4397,0.270318
home,17115,5901,0.344785
services,12915,4539,0.351452
electronics,17096,6193,0.362248


In [9]:
ranked = orders.sort_values('order_timestamp').copy()
ranked['order_number'] = ranked.groupby('user_id').cumcount()+1
ranked['next_order'] = ranked.groupby('user_id').order_timestamp.shift(-1)
first = ranked[(ranked.order_number==1) & (ranked.order_timestamp <= '2024-05-01')].copy()
first['repeat_60d'] = first.next_order.le(first.order_timestamp + pd.Timedelta(days=60))
first.repeat_60d.mean()

0.3544947904046523

## 6. Decision

1. Diagnose and test the mobile checkout gap first because it occurs closest to value and has a direct interface hypothesis.
2. Add channel-quality metrics (30-day purchase rate and revenue per acquired user) to acquisition reviews; volume alone is misleading.
3. Treat deeper early exploration as a candidate behavioral signal, not an operational activation metric.

The proposed experiment in the README randomizes decision-support prompts to test whether helping first-session users compare relevant offers changes subsequent 14-day purchasing. Intention-to-treat analysis separates the effect of offering the feature from the self-selected behavior of using it.